# Tableau Cloud — Commercial Finance Inventory Explorer
 **Lightweight replacement for the long-running Word-documentation pipeline.** Discover all accessible projects with names containing `Commercial Finance` (case-insensitive), include descendants, and inventory workbooks, published data sources and Tableau Prep flows. Export one formatted Excel file for exploration.

 **No Word document, no workbook downloads, no workbook XML parsing, no KPI or field-level metadata.** Uses Tableau PAT authentication and the project discovery architecture from your prior successful notebook.

**Enrichment:** Optional connection mode, modification timestamps, extract refresh, lifetime view usage and dated 30-day usage (with explicit availability/source statuses).

**Navigation enhancement:** Bidirectional Excel links and Tableau Cloud links, with project-based navigation and view-traffic cross-references.

**Viewer enrichment:** Optional Admin Insights CSV inputs identify preferred non-owner Creator/Explorer viewers per workbook, preserving the existing Excel sheets and hyperlinks.

## Step 1 — Install dependencies
 Run once in the notebook environment. The Excel file is authored with `openpyxl`. The Excel row/column outlines are created natively using openpyxl.

In [ ]:
%pip install -q requests pandas lxml openpyxl

## Step 2 — Imports and configuration
 Set the Tableau server, site, PAT environment-variable names, search phrase, output path, and whether to fetch optional high-level asset details. `fetch_asset_details=False` is fastest. No credentials are stored in the notebook.

In [ ]:
import os, re, json, csv, io, time, hashlib, zipfile, logging, getpass, traceback
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, Counter
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.formatting.rule import FormulaRule
from openpyxl.utils import get_column_letter
from typing import Any, Dict, List, Optional, Tuple
import requests
import pandas as pd
from lxml import etree
# Inventory-only pipeline: Word generation removed
# from docx import Document
# from docx.shared import Inches, Pt, RGBColor
# from docx.enum.text import WD_ALIGN_PARAGRAPH
# from docx.oxml import OxmlElement
# from docx.oxml.ns import qn

CONFIG = {
    # Existing successful Tableau Cloud setup
    "tableau_server": "https://your-tableau-pod.online.tableau.com",
    "site_content_url": "demo-site",
    "api_version": "3.29",  # Change if your site requires a newer REST API version
    "pat_name_env": "TABLEAU_PAT_NAME",
    "pat_secret_env": "TABLEAU_PAT_SECRET",

    # Scope discovery
    "project_search": "Commercial Finance",
    "include_descendant_projects": True,
    "dry_run": False,  # Set True to inventory matching projects and asset counts without downloads
    "max_assets_per_type": None,  # Optional pilot cap PER TYPE; None = all

    # Metadata coverage
    "query_metadata_api": False,
    "download_workbooks": False,
    "download_datasources": False,
    "download_flows": False,
    "parse_workbook_xml": False,
    "parse_datasource_xml": False,
    "include_extract": False,
    "capture_view_screenshots": False,
    "max_screenshots_per_workbook": None,
    "capture_flow_runs": False,
    "capture_flow_tasks": False,

    # Local outputs
    "output_dir": "commercial_finance_inventory_explorer",
    "inventory_filename": "Commercial_Finance_Project_Inventory.xlsx",
    "fetch_asset_details": False,
    # Optional enrichment: REST metadata, view traffic, GraphQL extract timestamps.
    "enrich_connection_mode": True,
    "enrich_last_modified": True,
    "enrich_extract_refresh": True,
    "enrich_lifetime_views": True,
    "enrich_30day_views": True,
    "admin_insights_traffic_csv": "",  # Optional locally exported dated view-event CSV
    "traffic_date_column": "Event Date",
    "traffic_workbook_id_column": "Workbook ID",
    "traffic_view_id_column": "View ID",
    "traffic_count_column": "",  # Blank = one event per row
    "max_enrichment_assets": None,  # None = all, integer for pilot
    "enrichment_pause_seconds": 0.0,
    "fetch_asset_details": False,  # Optional extra REST calls for high-level asset details
    # Activity prioritization (UTC); unknown traffic is NEVER treated as zero.
    "enable_activity_prioritization": True,
    "recent_activity_days": 30,
    "recent_modification_days": 30,
    "high_traffic_30day_threshold": 100,
    "medium_traffic_30day_threshold": 10,
    "high_lifetime_traffic_threshold": 1000,
    "priority_sheet_name": "Activity Priorities",
    "enable_excel_hyperlinks": True,
    "enable_tableau_cloud_links": True,
    "enable_view_traffic_links": True,
    # Viewer identification: Tableau Admin Insights TS Events + TS Users exports.
    # Use complete, unaggregated view-access events with actor identity.
    "viewer_events_csv": "",  # TS Events export, one event per row
    "viewer_users_csv": "",   # TS Users export with User ID and User Site Role
    "viewer_role_priority": ("Creator", "Explorer"),
    "viewer_allow_other_roles": False,  # False = do not select Viewer-role accounts
    "viewer_require_verified_workbook_id": True,
    "viewer_event_types": ("Access",),
    "viewer_event_name_keywords": ("access view", "view access", "view opened", "view accessed"),
    "viewer_allow_access_type_without_event_name": True,
    "viewer_lookback_days": None,  # None = all events present in export
    "prioritize_verified_non_owner_views": True,
    "non_owner_view_lookback_days": 30,
    "non_owner_high_traffic_threshold": 100,
    "non_owner_medium_traffic_threshold": 10,
    "non_owner_require_complete_coverage_for_zero": True,
    "non_owner_events_coverage_confirmed": False,  # Set True ONLY when TS Events export covers the full last 30 days


    "group_rows_by_project": True,
    "group_technical_columns": True,
    "export_csv": False,
    "export_excel": True,
    "export_json": False,
    "report_table_max_rows": None,  # None = ALL records, may produce a very long document
    "max_cell_chars": 1600,
    "fail_fast": False,
    "request_timeout": 120,
    "page_size": 1000,

    # Flags required by the reused TWBExtractor
    "parse_twb": True,
    "parse_datasources": True,
    "parse_worksheets": True,
    "parse_dashboards": True,
    "parse_calculations": True,
    "parse_parameters": True,
    "parse_filters": True,
    "parse_metadata_records": True,
}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ROOT = Path(CONFIG["output_dir"]).expanduser().resolve()
ROOT.mkdir(parents=True, exist_ok=True)
(RAW_DIR := ROOT / "raw").mkdir(exist_ok=True)
(CSV_DIR := ROOT / "csv").mkdir(exist_ok=True)
(SHOT_DIR := ROOT / "screenshots").mkdir(exist_ok=True)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("commercial_finance")

## Step 3 — Shared helper functions
 Retains name normalization, REST response helpers, and logging utilities from the earlier extraction notebook.

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def normalize_name(value: Any) -> str:
    """
    Normalize names for diagnostic / hierarchy matching.

    Does NOT replace exact matching when Tableau itself can perform
    an exact filter.
    """
    if value is None:
        return ""

    value = str(value)

    value = unicodedata.normalize("NFKC", value)
    value = re.sub(r"\s+", " ", value.strip())

    return value.casefold()

def safe_filename(value: str, max_length: int = 180) -> str:
    value = str(value)

    value = re.sub(r'[<>:"/\\|?*\x00-\x1F]', "_", value)
    value = value.strip().rstrip(".")

    if not value:
        value = "output"

    return value[:max_length]

def dataframe_from_records(records: Any) -> pd.DataFrame:
    if records is None:
        return pd.DataFrame()

    if isinstance(records, pd.DataFrame):
        return records.copy()

    if isinstance(records, list):
        if not records:
            return pd.DataFrame()

        return pd.json_normalize(records)

    if isinstance(records, dict):
        return pd.json_normalize([records])

    return pd.DataFrame()

def xml_string(element) -> Optional[str]:
    if not CONFIG["keep_raw_xml"]:
        return None

    try:
        return etree.tostring(
            element,
            encoding="unicode",
            pretty_print=False,
        )
    except Exception:
        return None

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()

def strip_namespace(tag: str) -> str:
    if not tag:
        return ""

    if "}" in tag:
        return tag.split("}", 1)[1]

    return tag

def element_attr(element, *names, default=None):
    for name in names:
        value = element.get(name)

        if value is not None:
            return value

    return default

def local_name(element) -> str:
    return strip_namespace(element.tag)

def load_twb_from_path(
    path: Path,
) -> Tuple[Path, bytes, etree._ElementTree, Dict[str, Any]]:

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    suffix = path.suffix.lower()

    if suffix == ".twb":

        xml_bytes = path.read_bytes()

        xml_root = etree.fromstring(xml_bytes)

        tree = etree.ElementTree(xml_root)

        info = {
            "source_path": str(path),
            "source_type": "twb",
            "sha256": sha256_file(path),
            "twb_member": path.name,
        }

        return path, xml_bytes, tree, info

    if suffix == ".twbx":

        with zipfile.ZipFile(path, "r") as z:

            members = z.namelist()

            twb_members = [
                m
                for m in members
                if m.lower().endswith(".twb")
            ]

            if not twb_members:
                raise RuntimeError(
                    f"No .twb file was found inside:\n{path}"
                )

            # Prefer a root-level TWB if available.
            twb_members.sort(
                key=lambda x: (
                    "/" in x,
                    x.count("/"),
                    x.lower(),
                )
            )

            twb_member = twb_members[0]

            xml_bytes = z.read(twb_member)

        xml_root = etree.fromstring(xml_bytes)

        tree = etree.ElementTree(xml_root)

        info = {
            "source_path": str(path),
            "source_type": "twbx",
            "sha256": sha256_file(path),
            "twb_member": twb_member,
        }

        return path, xml_bytes, tree, info

    raise ValueError(
        f"Unsupported Tableau workbook type: {suffix}"
    )

## Step 4 — Tableau Cloud PAT REST client
 The existing REST client is reused for authentication, pagination and project discovery. This inventory does not invoke deep GraphQL metadata or workbook downloads.

In [ ]:
class TableauRestClient:

    def __init__(
        self,
        server: str,
        site_content_url: str,
        api_version: str,
        pat_name: str,
        pat_secret: str,
        timeout: int = 120,
    ):
        self.server = server.rstrip("/")
        self.site_content_url = site_content_url
        self.api_version = api_version

        self.pat_name = pat_name
        self.pat_secret = pat_secret

        self.timeout = timeout

        self.session = requests.Session()

        self.site_id = None
        self.user_id = None
        self.auth_token = None

    # --------------------------------------------------------
    # URL HELPERS
    # --------------------------------------------------------

    @property
    def rest_base(self):
        return f"{self.server}/api/{self.api_version}"

    @property
    def metadata_url(self):
        # IMPORTANT:
        # Tableau Metadata API endpoint is /api/metadata/graphql
        return f"{self.server}/api/metadata/graphql"

    # --------------------------------------------------------
    # REST REQUEST
    # --------------------------------------------------------

    def _headers(self):
        if not self.auth_token:
            raise RuntimeError("Tableau client is not authenticated.")

        return {
            "X-Tableau-Auth": self.auth_token,
            "Accept": "application/json",
        }

    def get(
        self,
        path: str,
        params: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:
        url = f"{self.server}/api/{self.api_version}{path}"

        response = self.session.get(
            url,
            headers=self._headers(),
            params=params,
            timeout=self.timeout,
        )

        if response.status_code >= 400:
            raise RuntimeError(
                "Tableau REST API GET failed.\n"
                f"HTTP Status: {response.status_code}\n"
                f"URL: {response.url}\n"
                f"Response: {response.text[:2000]}"
            )

        if not response.text.strip():
            return {}

        return response.json()
    # --------------------------------------------------------
    # SIGN IN
    # --------------------------------------------------------

    def signin(self):

        url = f"{self.rest_base}/auth/signin"

        payload = {
            "credentials": {
                "personalAccessTokenName": self.pat_name,
                "personalAccessTokenSecret": self.pat_secret,
                "site": {
                    "contentUrl": self.site_content_url
                },
            }
        }

        response = self.session.post(
            url,
            json=payload,
            headers={
                "Accept": "application/json",
                "Content-Type": "application/json",
            },
            timeout=self.timeout,
        )

        if response.status_code >= 400:
            raise RuntimeError(
                "Tableau sign-in failed.\n\n"
                f"HTTP Status: {response.status_code}\n"
                f"URL: {response.url}\n"
                f"Response:\n{response.text[:5000]}"
            )

        data = response.json()

        credentials = data.get("credentials", {})

        self.auth_token = credentials.get("token")

        site = credentials.get("site", {})

        self.site_id = site.get("id")
        self.site_content_url = site.get(
            "contentUrl",
            self.site_content_url,
        )

        self.user_id = credentials.get("user", {}).get("id")

        if not self.auth_token:
            raise RuntimeError(
                "Tableau sign-in response did not contain an authentication token."
            )

        if not self.site_id:
            raise RuntimeError(
                "Tableau sign-in response did not contain a site ID."
            )

        print("Tableau sign-in successful.")
        print(f"  Site ID: {self.site_id}")
        print(f"  User ID: {self.user_id}")

    # --------------------------------------------------------
    # SIGN OUT
    # --------------------------------------------------------

    def signout(self):

        if not self.auth_token:
            return

        url = f"{self.rest_base}/auth/signout"

        try:
            response = self.session.post(
                url,
                headers=self._headers(),
                timeout=self.timeout,
            )

            if response.status_code not in {200, 204}:
                raise RuntimeError(
                    f"HTTP {response.status_code}: {response.text[:1000]}"
                )

            print("Tableau sign-out successful.")

        finally:
            self.auth_token = None
            self.site_id = None
            self.user_id = None

    # --------------------------------------------------------
    # RESPONSE LIST NORMALIZER
    # --------------------------------------------------------

    @staticmethod
    def _extract_list(
        container: Any,
        singular_key: Optional[str] = None,
    ) -> List[Dict[str, Any]]:

        if container is None:
            return []

        if isinstance(container, list):
            return container

        if isinstance(container, dict):

            if singular_key:
                value = container.get(singular_key)

                if value is None:
                    return []

                if isinstance(value, list):
                    return value

                if isinstance(value, dict):
                    return [value]

                return []

            # Generic dictionary containing one list-like value.
            for value in container.values():

                if isinstance(value, list):
                    return value

                if isinstance(value, dict):
                    return [value]

            return []

        return []

    # --------------------------------------------------------
    # PAGINATION
    # --------------------------------------------------------

    def _get_all_paginated(
        self,
        endpoint: str,
        container_key: str,
        singular_key: str,
        page_size: int = 1000,
        extra_params: Optional[Dict[str, Any]] = None,
    ) -> List[Dict[str, Any]]:

        if page_size < 1 or page_size > 1000:
            raise ValueError("page_size must be between 1 and 1000.")

        extra_params = extra_params or {}

        all_items = []

        page_number = 1
        total_available = None

        while True:

            params = dict(extra_params)

            params["pageSize"] = page_size
            params["pageNumber"] = page_number

            data = self.get(
                endpoint,
                params=params,
            )

            container = data.get(container_key, {})

            items = self._extract_list(
                container,
                singular_key=singular_key,
            )

            all_items.extend(items)

            pagination = data.get("pagination", {})

            if total_available is None:
                try:
                    total_available = int(
                        pagination.get("totalAvailable")
                    )
                except Exception:
                    total_available = None

            if total_available is not None:

                if len(all_items) >= total_available:
                    break

            if len(items) < page_size:
                break

            page_number += 1

        return all_items

    # --------------------------------------------------------
    # PROJECTS
    # --------------------------------------------------------

    def get_projects(self) -> List[Dict[str, Any]]:

        endpoint = f"/sites/{self.site_id}/projects"

        return self._get_all_paginated(
            endpoint=endpoint,
            container_key="projects",
            singular_key="project",
            page_size=1000,
        )

    # --------------------------------------------------------
    # PROJECT HIERARCHY
    # --------------------------------------------------------

    def _project_parent_id(self, project: Dict[str, Any]):
        parent_id = project.get("parentProjectId")

        if parent_id in {"", None}:
            return None

        return parent_id

    def _build_project_path(
        self,
        project: Dict[str, Any],
        projects_by_id: Dict[str, Dict[str, Any]],
    ) -> List[str]:

        names = []

        current = project

        visited = set()

        while current:

            project_id = current.get("id")

            if project_id in visited:
                raise RuntimeError(
                    "Cycle detected while reconstructing Tableau project hierarchy."
                )

            visited.add(project_id)

            names.append(current.get("name", ""))

            parent_id = self._project_parent_id(current)

            if not parent_id:
                break

            current = projects_by_id.get(parent_id)

            if current is None:
                raise RuntimeError(
                    "Unable to resolve parent project while reconstructing "
                    f"hierarchy.\n"
                    f"Project: {project.get('name')}\n"
                    f"Project ID: {project_id}\n"
                    f"Missing parentProjectId: {parent_id}"
                )

        names.reverse()

        return names

    def resolve_project_path(
        self,
        project_path: List[str],
    ) -> Dict[str, Any]:

        if not project_path:
            raise ValueError("project_path cannot be empty.")

        print()
        print("Resolving Tableau project hierarchy...")
        print(
            "  Requested: "
            + " / ".join(project_path)
        )

        projects = self.get_projects()

        if not projects:
            raise RuntimeError(
                "No accessible Tableau projects were returned by the REST API."
            )

        projects_by_id = {
            p.get("id"): p
            for p in projects
            if p.get("id")
        }

        # ----------------------------------------------------
        # Walk hierarchy one level at a time.
        # ----------------------------------------------------

        parent_id = None
        resolved_nodes = []

        for level_index, requested_name in enumerate(project_path):

            normalized_requested = normalize_name(requested_name)

            candidates = []

            for project in projects:

                if normalize_name(project.get("name")) != normalized_requested:
                    continue

                candidate_parent = self._project_parent_id(project)

                if candidate_parent == parent_id:
                    candidates.append(project)

            if not candidates:

                available = []

                for project in projects:

                    candidate_parent = self._project_parent_id(project)

                    if candidate_parent == parent_id:
                        available.append(
                            {
                                "id": project.get("id"),
                                "name": project.get("name"),
                            }
                        )

                available = sorted(
                    available,
                    key=lambda x: normalize_name(x["name"]),
                )

                raise RuntimeError(
                    "Could not resolve Tableau project hierarchy.\n\n"
                    f"Level: {level_index + 1}\n"
                    f"Requested project: {requested_name!r}\n"
                    f"Expected parent ID: {parent_id!r}\n\n"
                    "Available projects at this hierarchy level:\n"
                    + "\n".join(
                        f"  - {x['name']} | ID={x['id']}"
                        for x in available[:100]
                    )
                )

            if len(candidates) > 1:

                details = "\n".join(
                    f"  - {p.get('name')} | "
                    f"ID={p.get('id')} | "
                    f"parentProjectId={p.get('parentProjectId')}"
                    for p in candidates
                )

                raise RuntimeError(
                    "Ambiguous Tableau project hierarchy.\n\n"
                    f"Multiple projects named {requested_name!r} "
                    f"exist under parent ID {parent_id!r}:\n\n"
                    f"{details}\n\n"
                    "The pipeline will not guess which project is correct."
                )

            selected = candidates[0]

            resolved_nodes.append(selected)

            parent_id = selected.get("id")

        final_project = resolved_nodes[-1]

        reconstructed_path = self._build_project_path(
            final_project,
            projects_by_id,
        )

        expected_normalized = [
            normalize_name(x)
            for x in project_path
        ]

        actual_normalized = [
            normalize_name(x)
            for x in reconstructed_path
        ]

        if expected_normalized != actual_normalized:
            raise RuntimeError(
                "Resolved Tableau project hierarchy does not match the "
                "requested hierarchy.\n\n"
                f"Requested: {' / '.join(project_path)}\n"
                f"Resolved:  {' / '.join(reconstructed_path)}"
            )

        print(
            "  Resolved: "
            + " / ".join(reconstructed_path)
        )

        print(
            f"  Final Project ID: {final_project.get('id')}"
        )

        return {
            "project": final_project,
            "nodes": resolved_nodes,
            "path": reconstructed_path,
            "all_projects": projects,
        }

    # --------------------------------------------------------
    # EXACT WORKBOOK NAME
    # --------------------------------------------------------

    def get_workbooks_by_exact_name(
        self,
        workbook_name: str,
    ) -> List[Dict[str, Any]]:

        print()
        print("Locating workbook by exact name...")
        print(f"  Workbook: {workbook_name}")

        endpoint = f"/sites/{self.site_id}/workbooks"

        # ----------------------------------------------------
        # FIRST ATTEMPT:
        # Tableau server-side exact-name filter.
        # ----------------------------------------------------

        filtered = self._get_all_paginated(
            endpoint=endpoint,
            container_key="workbooks",
            singular_key="workbook",
            page_size=1000,
            extra_params={
                "filter": f"name:eq:{workbook_name}",
            },
        )

        # Defensive exact comparison.
        exact = [
            wb
            for wb in filtered
            if wb.get("name") == workbook_name
        ]

        if exact:
            return exact

        # ----------------------------------------------------
        # FALLBACK:
        # Full paginated scan.
        #
        # This protects us if a server-side filter behaves
        # unexpectedly or if the workbook name has a subtle
        # representation issue.
        # ----------------------------------------------------

        print(
            "  Exact-name filter returned no result; "
            "performing full paginated workbook scan..."
        )

        all_workbooks = self._get_all_paginated(
            endpoint=endpoint,
            container_key="workbooks",
            singular_key="workbook",
            page_size=1000,
        )

        exact = [
            wb
            for wb in all_workbooks
            if wb.get("name") == workbook_name
        ]

        if exact:
            return exact

        # ----------------------------------------------------
        # DIAGNOSTICS:
        # normalized near matches.
        # ----------------------------------------------------

        requested_normalized = normalize_name(workbook_name)

        near_matches = []

        for wb in all_workbooks:

            actual_name = wb.get("name", "")

            if normalize_name(actual_name) == requested_normalized:

                near_matches.append(wb)

        if near_matches:

            print()
            print(
                "  WARNING: No exact string match was found, "
                "but normalized-name matches exist:"
            )

            for wb in near_matches[:50]:
                project = wb.get("project", {})

                print(
                    f"    - {wb.get('name')!r} | "
                    f"Workbook ID={wb.get('id')} | "
                    f"Project={project.get('name')} | "
                    f"Project ID={project.get('id')}"
                )

        return []

    # --------------------------------------------------------
    # WORKBOOK SELECTION
    # --------------------------------------------------------

    def get_workbook(
        self,
        workbook_name: Optional[str] = None,
        workbook_luid: Optional[str] = None,
        project_path: Optional[List[str]] = None,
    ) -> Dict[str, Any]:

        workbook_name = (
            workbook_name
            or CONFIG.get("workbook_name")
        )

        workbook_luid = (
            workbook_luid
            or CONFIG.get("workbook_luid")
            or ""
        ).strip()

        project_path = (
            project_path
            if project_path is not None
            else CONFIG.get("project_path")
        )

        resolved_project = None

        if project_path:

            resolved_project = self.resolve_project_path(
                project_path
            )

            target_project = resolved_project["project"]

        else:
            target_project = None

        # ----------------------------------------------------
        # DIRECT LUID LOOKUP
        # ----------------------------------------------------

        if workbook_luid:

            print()
            print("Locating workbook by supplied LUID...")
            print(f"  Workbook LUID: {workbook_luid}")

            endpoint = (
                f"/sites/{self.site_id}/workbooks/{workbook_luid}"
            )

            data = self.get(endpoint)

            workbook = data.get("workbook")

            if not workbook:
                raise RuntimeError(
                    f"Tableau returned no workbook for LUID: "
                    f"{workbook_luid}"
                )

            if target_project:

                actual_project = workbook.get("project", {})

                actual_project_id = actual_project.get("id")
                expected_project_id = target_project.get("id")

                if actual_project_id != expected_project_id:

                    actual_path = (
                        actual_project.get("name")
                        or "<unknown>"
                    )

                    raise RuntimeError(
                        "Supplied workbook LUID does not belong to "
                        "the configured project hierarchy.\n\n"
                        f"Workbook LUID: {workbook_luid}\n"
                        f"Workbook: {workbook.get('name')}\n"
                        f"Actual project: {actual_path}\n"
                        f"Actual project ID: {actual_project_id}\n"
                        f"Expected project path: "
                        f"{' / '.join(project_path)}\n"
                        f"Expected project ID: {expected_project_id}"
                    )

            print("  Workbook found by LUID.")

            return workbook

        # ----------------------------------------------------
        # NAME + PROJECT HIERARCHY
        # ----------------------------------------------------

        if not workbook_name:
            raise RuntimeError(
                "No workbook_name or workbook_luid was supplied."
            )

        candidates = self.get_workbooks_by_exact_name(
            workbook_name
        )

        if not candidates:

            raise RuntimeError(
                "No workbook with the exact name was found.\n\n"
                f"Workbook name: {workbook_name}\n"
                f"Expected project hierarchy: "
                f"{' / '.join(project_path or [])}\n\n"
                "The pipeline intentionally does not guess from "
                "near-name matches."
            )

        # ----------------------------------------------------
        # PROJECT FILTER
        # ----------------------------------------------------

        if target_project:

            expected_project_id = target_project.get("id")

            project_candidates = []

            for wb in candidates:

                wb_project = wb.get("project", {})

                if wb_project.get("id") == expected_project_id:
                    project_candidates.append(wb)

            candidates = project_candidates

        # ----------------------------------------------------
        # ZERO MATCHES AFTER PROJECT FILTER
        # ----------------------------------------------------

        if not candidates:

            raise RuntimeError(
                "Workbook name was found, but no matching workbook "
                "exists in the requested project hierarchy.\n\n"
                f"Workbook name: {workbook_name}\n"
                f"Expected project hierarchy: "
                f"{' / '.join(project_path or [])}\n\n"
                "Exact-name workbook candidates returned by Tableau:"
            )

        # ----------------------------------------------------
        # MULTIPLE MATCHES
        # ----------------------------------------------------

        if len(candidates) > 1:

            details = []

            for wb in candidates:

                project = wb.get("project", {})

                details.append(
                    f"  - Workbook: {wb.get('name')}\n"
                    f"    LUID: {wb.get('id')}\n"
                    f"    Project: {project.get('name')}\n"
                    f"    Project ID: {project.get('id')}\n"
                    f"    Owner: {wb.get('owner', {}).get('name')}"
                )

            raise RuntimeError(
                "Multiple workbooks match the exact workbook name "
                "and project hierarchy.\n\n"
                + "\n".join(details)
                + "\n\n"
                "Set CONFIG['workbook_luid'] explicitly so the "
                "pipeline does not guess."
            )

        workbook = candidates[0]

        print()
        print("Workbook selected successfully.")
        print(f"  Name: {workbook.get('name')}")
        print(f"  LUID: {workbook.get('id')}")

        project = workbook.get("project", {})

        print(f"  Project: {project.get('name')}")
        print(f"  Project ID: {project.get('id')}")

        owner = workbook.get("owner", {})

        print(f"  Owner: {owner.get('name')}")

        return workbook

    # --------------------------------------------------------
    # DOWNLOAD WORKBOOK
    # --------------------------------------------------------

    def download_cloud_workbook(
        self,
        workbook: Dict[str, Any],
    ) -> Path:

        workbook_id = workbook.get("id")

        if not workbook_id:
            raise RuntimeError(
                "Selected workbook does not contain a REST workbook ID/LUID."
            )

        work_dir = WORK_DIR
        work_dir.mkdir(parents=True, exist_ok=True)

        safe_name = safe_filename(
            workbook.get("name", "tableau_workbook")
        )

        print()
        print("Downloading Tableau workbook...")
        print(f"  Destination directory: {work_dir}")
        print(f"  Include extract: {CONFIG['include_extract']}")

        # Tableau can return either a TWB (XML) or TWBX (ZIP).
        # Try compatible Accept headers if a gateway rejects content negotiation.
        base_url = (
            f"{self.rest_base}/sites/{self.site_id}/"
            f"workbooks/{workbook_id}/content"
        )
        auth_header = {"X-Tableau-Auth": self.auth_token}
        include_extract = "true" if CONFIG["include_extract"] else "false"

        attempts = [
            ({"Accept": "*/*"}, {"includeExtract": include_extract}),
            ({"Accept": "application/octet-stream"}, {"includeExtract": include_extract}),
            ({"Accept": "application/xml"}, {"includeExtract": include_extract}),
            ({"Accept": "*/*"}, None),
        ]

        response = None
        last_error = None

        for accept_headers, params in attempts:
            headers = {**auth_header, **accept_headers}
            try:
                candidate = self.session.get(
                    base_url,
                    headers=headers,
                    params=params,
                    timeout=self.timeout,
                    allow_redirects=True,
                )
            except requests.RequestException as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                continue

            if candidate.status_code < 400 and candidate.content:
                response = candidate
                break

            response_summary = (
                f"HTTP {candidate.status_code}; "
                f"Content-Type={candidate.headers.get('Content-Type')!r}; "
                f"URL={candidate.url}; "
                f"Body={candidate.text[:1000]!r}"
            )
            last_error = response_summary

            # Retry only for 406. Other statuses should be surfaced directly.
            if candidate.status_code != 406:
                raise RuntimeError(
                    "Tableau workbook download failed.\n" + response_summary
                )

        if response is None:
            raise RuntimeError(
                "Tableau workbook download failed after content-negotiation retries.\n"
                f"Last response/error: {last_error or 'No response received.'}"
            )

        content = response.content
        if not content:
            raise RuntimeError("Tableau workbook download returned an empty response.")

        # Detect actual format; includeExtract=False can yield a .twb rather than .twbx.
        trimmed = content.lstrip(b"\xef\xbb\xbf\x00\t\r\n ")
        if content.startswith(b"PK\x03\x04"):
            extension = ".twbx"
        elif (
            trimmed.startswith(b"<?xml")
            or trimmed.startswith(b"<workbook")
            or b"<workbook" in trimmed[:1000]
        ):
            extension = ".twb"
        else:
            content_type = response.headers.get("Content-Type", "")
            raise RuntimeError(
                "Tableau returned a successful response, but it does not look like "
                "a TWB XML document or TWBX ZIP archive.\n"
                f"Content-Type: {content_type!r}\n"
                f"First 200 bytes: {content[:200]!r}"
            )

        output_path = work_dir / f"{safe_name}{extension}"
        output_path.write_bytes(content)

        print(f"  Downloaded: {output_path.stat().st_size:,} bytes")
        print(f"  Detected format: {extension[1:].upper()}")
        print(f"  Final URL: {response.url}")

        return output_path

    # --------------------------------------------------------
    # WORKBOOK VIEWS / SCREENSHOT EXPORT
    # --------------------------------------------------------

    def get_workbook_views(self, workbook_id: str) -> List[Dict[str, Any]]:
        """Return all views belonging to a workbook using REST pagination."""
        endpoint = f"/sites/{self.site_id}/workbooks/{workbook_id}/views"
        return self._get_all_paginated(
            endpoint=endpoint,
            container_key="views",
            singular_key="view",
            page_size=1000,
        )

    def download_view_image(self, view_id: str, output_path: Path) -> Path:
        """Download a rendered PNG for a Tableau view."""
        url = (
            f"{self.rest_base}/sites/{self.site_id}/views/"
            f"{view_id}/image"
        )
        params = {
            "maxAge": CONFIG.get("screenshot_max_age_minutes", 1),
            "vizWidth": CONFIG.get("screenshot_width", 1600),
            "vizHeight": CONFIG.get("screenshot_height", 900),
        }
        response = self.session.get(
            url,
            headers={
                "X-Tableau-Auth": self.auth_token,
                "Accept": "image/png,application/octet-stream,*/*",
            },
            params=params,
            timeout=self.timeout,
        )
        if response.status_code >= 400:
            raise RuntimeError(
                f"View image export failed for view {view_id}. "
                f"HTTP {response.status_code}: {response.text[:1000]}"
            )
        if not response.content:
            raise RuntimeError(f"View image export returned empty content for {view_id}.")
        content_type = response.headers.get("Content-Type", "").lower()
        if "json" in content_type or "xml" in content_type:
            raise RuntimeError(
                f"View image endpoint returned {content_type}, not an image."
            )
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_bytes(response.content)
        return output_path

    # --------------------------------------------------------
    # METADATA API
    # --------------------------------------------------------

    def metadata_query(
        self,
        query: str,
        variables: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:

        if not self.auth_token:
            raise RuntimeError(
                "Cannot query Metadata API before Tableau sign-in."
            )

        # ----------------------------------------------------
        # CRITICAL:
        # Correct Tableau Metadata API endpoint.
        #
        # NOT:
        #   /graphql
        #
        # NOT:
        #   /metadata/graphql
        #
        # YES:
        #   /api/metadata/graphql
        # ----------------------------------------------------

        url = self.metadata_url

        payload = {
            "query": query,
        }

        if variables is not None:
            payload["variables"] = variables

        response = self.session.post(
            url,
            headers={
                "X-Tableau-Auth": self.auth_token,
                "Accept": "application/json",
                "Content-Type": "application/json",
            },
            json=payload,
            timeout=self.timeout,
        )

        if response.status_code >= 400:
            raise RuntimeError(
                "Tableau Metadata API request failed.\n\n"
                f"HTTP Status: {response.status_code}\n"
                f"URL: {response.url}\n"
                f"Response:\n{response.text[:10000]}"
            )

        try:
            result = response.json()

        except Exception as exc:
            raise RuntimeError(
                "Tableau Metadata API returned a non-JSON response.\n\n"
                f"HTTP Status: {response.status_code}\n"
                f"URL: {response.url}\n"
                f"Response:\n{response.text[:10000]}"
            ) from exc

        if result.get("errors"):

            raise RuntimeError(
                "Tableau Metadata API returned GraphQL errors.\n\n"
                f"{json.dumps(result['errors'], indent=2, ensure_ascii=False)}"
            )

        return result.get("data", result)

    # --------------------------------------------------------
    # WORKBOOK METADATA QUERY
    # --------------------------------------------------------

    def query_workbook_metadata(
        self,
        workbook_luid: str,
    ) -> Dict[str, Any]:

        if not workbook_luid:
            raise ValueError(
                "workbook_luid is required for Metadata API extraction."
            )

        print()
        print("Querying Tableau Metadata API...")
        print(f"  Workbook LUID: {workbook_luid}")
        print(f"  Endpoint: {self.metadata_url}")

        # ----------------------------------------------------
        # The Metadata API supports filtering Workbook by LUID.
        #
        # This is deliberately NOT filtered by workbook name,
        # because duplicate workbook names exist on the site.
        # ----------------------------------------------------

        query = """
        query WorkbookMetadata($workbookLuid: String!) {
          workbooks(filter: {luid: $workbookLuid}) {
            id
            luid
            name
            containsUnsupportedCustomSql
            projectVizportalUrlId
            projectName
            projectLuid
            containerType
            containerName

            owner {
              id
              luid
              username
              name
              domain
              email
            }

            tags {
              id
              name
            }

            sheets {
              id
              luid
              name
              documentViewId
              path
              createdAt
              updatedAt
              index
              tags {
                id
                name
              }

              worksheetFields {
                id
                name
                fullyQualifiedName
                description
                aggregation
                aggregationParam
                formula
                isAutoGenerated
                hasUserReference
              }

              datasourceFields {
                id
                name
              }
            }

            dashboards {
              id
              luid
              name
              documentViewId
              path
              createdAt
              updatedAt
              index
              tags {
                id
                name
              }
            }

            embeddedDatasources {
              id
              name
              hasUserReference
              hasExtracts
              containsUnsupportedCustomSql

              fields {
                id
                name
              }
            }

            parameters {
              id
              name
            }
          }
        }
        """

        data = self.metadata_query(
            query=query,
            variables={
                "workbookLuid": workbook_luid,
            },
        )

        workbooks = data.get("workbooks", [])

        if not workbooks:

            raise RuntimeError(
                "Metadata API returned no workbook for the selected "
                f"REST workbook LUID: {workbook_luid}\n\n"
                "Possible causes:\n"
                "  1. Metadata API / Tableau Catalog is not enabled.\n"
                "  2. The authenticated PAT/user lacks required access.\n"
                "  3. Tableau metadata indexing has not caught up yet.\n"
                "  4. The workbook is not visible to the authenticated user."
            )

        if len(workbooks) > 1:

            raise RuntimeError(
                "Metadata API returned more than one workbook for "
                f"LUID {workbook_luid}, which should not happen."
            )

        workbook_metadata = workbooks[0]

        print("Metadata API query successful.")

        return workbook_metadata

## Step 5 — Recursive project discovery and asset inventory
 Find matching projects, reconstruct paths and include descendant folders. Fetch the site's accessible workbooks, data sources and flows, then filter them by included project IDs.

In [ ]:
def nested_id(item, key):
    obj = item.get(key)
    if isinstance(obj, dict):
        return str(obj.get("id") or obj.get("luid") or "")
    return str(item.get(key + "Id") or obj or "")


def project_id_of(item):
    return nested_id(item, "project") or str(item.get("projectId") or "")


def all_pages(client, endpoint, container, singular, allow_unsupported=False):
    try:
        return client._get_all_paginated(endpoint, container, singular, page_size=CONFIG["page_size"])
    except Exception as exc:
        if allow_unsupported:
            log.warning("Optional endpoint unavailable %s: %s", endpoint, exc)
            ERRORS.append({"stage":"optional_endpoint", "asset_type":container, "asset_id":"", "error":str(exc)})
            return []
        raise


def discover_projects(projects, phrase, descendants=True):
    by_id = {str(p.get("id")): p for p in projects if p.get("id")}
    children = defaultdict(list)
    for p in projects:
        parent = str(p.get("parentProjectId") or "")
        if parent: children[parent].append(str(p.get("id")))
    matches = {pid for pid,p in by_id.items() if phrase.casefold() in str(p.get("name", "")).casefold()}
    selected = set(matches)
    if descendants:
        queue = list(matches)
        while queue:
            for child in children.get(queue.pop(), []):
                if child not in selected:
                    selected.add(child); queue.append(child)
    def path_for(pid):
        path, seen = [], set()
        while pid in by_id and pid not in seen:
            seen.add(pid)
            p = by_id[pid]
            path.append(str(p.get("name") or ""))
            pid = str(p.get("parentProjectId") or "")
        return " / ".join(reversed(path))
    rows = []
    for pid in sorted(selected, key=path_for):
        row = dict(by_id[pid]); row["project_path"] = path_for(pid)
        row["direct_name_match"] = pid in matches
        rows.append(row)
    return rows, selected, matches


def get_assets(client, project_ids):
    site = client.site_id
    definitions = {
        "workbooks": (f"/sites/{site}/workbooks", "workbooks", "workbook"),
        "datasources": (f"/sites/{site}/datasources", "datasources", "datasource"),
        "flows": (f"/sites/{site}/flows", "flows", "flow"),
    }
    assets = {}
    for kind,(endpoint,container,singular) in definitions.items():
        rows = all_pages(client, endpoint, container, singular, allow_unsupported=(kind=="flows"))
        selected = [r for r in rows if project_id_of(r) in project_ids]
        cap = CONFIG["max_assets_per_type"]
        if cap is not None: selected = selected[:int(cap)]
        assets[kind] = selected
        log.info("%s: %d site-wide visible, %d selected", kind, len(rows), len(selected))
    return assets


def optional_get(client, endpoint, container=None, singular=None):
    try:
        if container:
            return all_pages(client, endpoint, container, singular, allow_unsupported=True)
        return client.get(endpoint)
    except Exception as exc:
        ERRORS.append({"stage":"optional_get", "asset_type":container or "detail", "asset_id":endpoint, "error":str(exc)})
        log.warning("Could not read optional detail %s: %s", endpoint, exc)
        return [] if container else {}


def download_binary(client, endpoint, target, params=None):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    url = client.rest_base + endpoint
    with client.session.get(url, headers=client._headers(), params=params, timeout=CONFIG["request_timeout"], stream=True) as response:
        if response.status_code >= 400:
            raise RuntimeError(f"HTTP {response.status_code}: {response.text[:600]}")
        with target.open("wb") as f:
            for chunk in response.iter_content(1024*1024):
                if chunk: f.write(chunk)
    return target


def record_error(stage, kind, asset, exc):
    ERRORS.append({"stage":stage,"asset_type":kind,"asset_id":asset.get("id", ""),"asset_name":asset.get("name", ""),"error":str(exc)})
    log.warning("%s / %s / %s: %s", stage, kind, asset.get("name", ""), exc)
    if CONFIG["fail_fast"]: raise exc


def add_rows(table, records, context):
    if isinstance(records, dict): records = [records]
    for record in records or []:
        if isinstance(record, dict): TABLES[table].append({**context, **record})
        else: TABLES[table].append({**context, "value":str(record)})


def flatten_twb_tables(twb_tables, context):
    for key, records in twb_tables.items():
        add_rows("workbook_xml_" + key, records, context)


def parse_datasource_package(path, context):
    # Published data sources can be .tds (XML) or .tdsx (ZIP with .tds).
    suffix = path.suffix.lower()
    if suffix == ".tds":
        raw = path.read_bytes()
    elif suffix == ".tdsx":
        with zipfile.ZipFile(path) as z:
            members = sorted([x for x in z.namelist() if x.lower().endswith(".tds")])
            if not members: raise RuntimeError("No .tds found inside published datasource package")
            raw = z.read(members[0])
    else: return
    root = etree.fromstring(raw)
    for ds in root.xpath("//*[local-name()='datasource']"):
        add_rows("datasource_xml_datasources", [{"name":ds.get("name"),"caption":ds.get("caption"),"version":ds.get("version")}], context)
        for el in ds.iter():
            tag = etree.QName(el).localname if isinstance(el.tag, str) else ""
            if tag in {"column","calculation","connection","relation","metadata-record","column-instance","aliases","group","set"}:
                rec = {"xml_tag":tag, **{str(k):str(v) for k,v in el.attrib.items()}}
                if tag == "calculation": rec["formula"] = el.get("formula", "")
                add_rows("datasource_xml_"+tag.replace("-","_"), [rec], context)


def asset_context(kind, asset, project_paths):
    pid = project_id_of(asset)
    return {"asset_type":kind, "asset_id":asset.get("id", ""), "asset_name":asset.get("name", ""),
            "project_id":pid, "project_path":project_paths.get(pid,""), "run_id":RUN_ID}

## Step 6 — Authenticate and discover the scope
 Use PAT name and secret from environment variables or local prompts. Lists the matching project paths and the number of visible assets before generating the Excel workbook.

In [ ]:
ERRORS = []
TABLES = defaultdict(list)
PAT_NAME = os.environ.get(CONFIG["pat_name_env"]) or input("Tableau PAT name: ").strip()
PAT_SECRET = os.environ.get(CONFIG["pat_secret_env"]) or getpass.getpass("Tableau PAT secret (hidden): ")
if not PAT_NAME or not PAT_SECRET:
    raise ValueError("Both PAT name and PAT secret are required.")
client = TableauRestClient(CONFIG["tableau_server"], CONFIG["site_content_url"],
                           CONFIG["api_version"], PAT_NAME, PAT_SECRET,
                           timeout=CONFIG["request_timeout"])
client.signin()
# Drop local references after login; client retains credentials for its existing sign-in implementation.
del PAT_SECRET
ALL_PROJECTS = client.get_projects()
MATCHED_PROJECTS, PROJECT_IDS, DIRECT_MATCH_IDS = discover_projects(
    ALL_PROJECTS, CONFIG["project_search"], CONFIG["include_descendant_projects"])
PROJECT_PATHS = {str(p["id"]):p["project_path"] for p in MATCHED_PROJECTS}
add_rows("projects", MATCHED_PROJECTS, {"run_id":RUN_ID})
print(f"Visible projects: {len(ALL_PROJECTS)}; direct name matches: {len(DIRECT_MATCH_IDS)}; included projects: {len(PROJECT_IDS)}")
if not MATCHED_PROJECTS:
    raise RuntimeError("No accessible projects matched the configured search phrase.")
display(pd.DataFrame(MATCHED_PROJECTS)[["name","project_path","direct_name_match","id"]])
ASSETS = get_assets(client, PROJECT_IDS)
for kind, rows in ASSETS.items():
    for item in rows: add_rows(kind, [item], asset_context(kind,item,PROJECT_PATHS))
print("Selected asset counts:", {k:len(v) for k,v in ASSETS.items()})

## Step 7 — Build lightweight inventory tables
 Produce project and asset records with their full project path, name, owner, timestamps, identifiers and available REST summary attributes. Optional detail requests can be enabled in configuration. No binary packages are downloaded.

## Step 7 — Enrich asset metadata and usage (optional)

This step uses Tableau REST to retrieve connection modes and lifetime **view** traffic, and attempts the Tableau Metadata API for extract last-update timestamps. `updatedAt` represents content modification, not data refresh. Workbook traffic is the sum of accessible underlying Tableau views (not unique visitors). A 30-day figure requires dated events from a locally exported **Admin Insights TS Events** CSV; the notebook does **not** derive 30-day usage from lifetime totals. Configure the CSV path and column names in Step 2. Unsupported or permission-restricted metrics are marked **Not Available**, never zero. Calls are read-only.

In [ ]:
from datetime import timedelta
from collections import defaultdict as _dd

NOT_AVAILABLE = "Not Available"
ENRICHED = {}
VIEW_TRAFFIC = []
TRAFFIC_30DAY = {}
ENRICHMENT_ERRORS = []


def enrich_error(kind, asset, stage, exc):
    msg = str(exc)
    rec = {"stage": stage, "asset_type": kind, "asset_id": str(asset.get("id", "")),
           "asset_name": asset.get("name", ""), "error": msg[:1200]}
    ENRICHMENT_ERRORS.append(rec)
    ERRORS.append(rec)
    log.warning("Enrichment %s %s: %s", kind, stage, msg[:200])


def rest_asset(kind, asset):
    # Query individual assets only when needed, avoiding binary downloads.
    endpoint = {"workbooks": "workbooks", "datasources": "datasources", "flows": "flows"}[kind]
    singular = {"workbooks": "workbook", "datasources": "datasource", "flows": "flow"}[kind]
    return client.get(f"/sites/{client.site_id}/{endpoint}/{asset['id']}").get(singular, {})


def graphql_metadata(query, variables=None):
    url = f"{client.server}/api/metadata/graphql"
    response = client.session.post(url, json={"query": query, "variables": variables or {}},
                                   headers={**client._headers(), "Content-Type": "application/json"},
                                   timeout=CONFIG["request_timeout"])
    response.raise_for_status()
    payload = response.json()
    if payload.get("errors"):
        raise RuntimeError(str(payload["errors"])[:1000])
    return payload.get("data") or {}


def extract_timestamp(kind, asset):
    # GraphQL metadata supports extractLastUpdateTime; REST updatedAt is NOT an extract timestamp.
    # Published data sources and workbook-embedded data sources use different metadata objects.
    if kind != "datasources":
        return NOT_AVAILABLE, "Not Available — no verified extract timestamp for this asset type"
    query = '''query ($luid: String!) {
      publishedDatasources(filter: {luid: $luid}) {
        luid hasExtracts extractLastUpdateTime
      }
    }'''
    try:
        data = graphql_metadata(query, {"luid": str(asset["id"])})
        items = data.get("publishedDatasources") or []
        exact = next((x for x in items if str(x.get("luid")) == str(asset["id"])), None)
        if exact:
            if exact.get("hasExtracts") is False:
                return NOT_AVAILABLE, "Not Applicable — Live"
            return exact.get("extractLastUpdateTime") or NOT_AVAILABLE, "Metadata API: extractLastUpdateTime"
        return NOT_AVAILABLE, "Metadata API — no matching published data source"
    except Exception as exc:
        enrich_error(kind, asset, "metadata_extract_timestamp", exc)
        return NOT_AVAILABLE, "Metadata API unavailable / insufficient permissions"


def connection_mode(kind, asset, detail):
    if kind == "flows":
        return NOT_AVAILABLE, "Not Applicable — Flow"
    flag = detail.get("hasExtracts", asset.get("hasExtracts"))
    if isinstance(flag, str):
        flag = flag.lower() == "true" if flag.lower() in ("true", "false") else None
    if flag is True:
        return "Extract", "REST hasExtracts"
    if flag is False and kind == "datasources":
        return "Live", "REST hasExtracts"
    # A workbook can contain multiple embedded/published connections.
    # Never claim Live for a workbook solely from hasExtracts=False.
    return "Unknown", "REST hasExtracts unavailable or mixed connection types"


def workbook_views(asset):
    endpoint = f"/sites/{client.site_id}/workbooks/{asset['id']}/views"
    views = client._get_all_paginated(endpoint, "views", "view",
                                      page_size=CONFIG["page_size"],
                                      extra_params={"includeUsageStatistics": "true"})
    count, observed = 0, 0
    for view in views:
        usage = view.get("usage") or {}
        raw = usage.get("totalViewCount")
        row = {"Workbook ID": str(asset["id"]), "Workbook Name": asset.get("name", ""),
               "View ID": str(view.get("id", "")), "View Name": view.get("name", ""),
               "Lifetime View Count": NOT_AVAILABLE}
        if raw is not None:
            try:
                val = int(raw)
                count += val
                observed += 1
                row["Lifetime View Count"] = val
            except (TypeError, ValueError):
                pass
        VIEW_TRAFFIC.append(row)
    if not views:
        return NOT_AVAILABLE, "REST: no accessible views"
    if observed != len(views):
        return NOT_AVAILABLE, f"REST: usage statistics missing for {len(views)-observed} of {len(views)} views"
    return count, f"REST: sum of {observed} accessible views' lifetime counts"


def load_30day_events():
    path = str(CONFIG.get("admin_insights_traffic_csv") or "").strip()
    if not path:
        return {}, "Not Available — Admin Insights event CSV not configured"
    df = pd.read_csv(Path(path).expanduser(), low_memory=False)
    date_col = CONFIG["traffic_date_column"]
    workbook_col = CONFIG["traffic_workbook_id_column"]
    view_col = CONFIG["traffic_view_id_column"]
    count_col = CONFIG["traffic_count_column"]
    if date_col not in df:
        raise ValueError(f"Missing CSV date column: {date_col}")
    if workbook_col not in df and view_col not in df:
        raise ValueError(f"CSV requires {workbook_col!r} or {view_col!r} for exact asset matching")
    if count_col and count_col not in df:
        raise ValueError(f"Missing CSV count column: {count_col}")
    # UTC half-open 30-day interval, counting dated events, not distinct users.
    now = datetime.now(timezone.utc)
    start = now - timedelta(days=30)
    timestamps = pd.to_datetime(df[date_col], errors="coerce", utc=True)
    df = df.loc[(timestamps >= start) & (timestamps <= now)].copy()
    if count_col:
        df["_events"] = pd.to_numeric(df[count_col], errors="coerce").fillna(0)
    else:
        df["_events"] = 1
    counts = _dd(float)
    if workbook_col in df:
        for key, val in df.groupby(df[workbook_col].astype(str))["_events"].sum().items():
            counts[str(key)] += float(val)
    else:
        view_to_workbook = {str(v["View ID"]): str(v["Workbook ID"]) for v in VIEW_TRAFFIC if v["View ID"]}
        for key, val in df.groupby(df[view_col].astype(str))["_events"].sum().items():
            if key in view_to_workbook:
                counts[view_to_workbook[key]] += float(val)
    # A zero is valid only if the CSV covers the entire 30-day period and all relevant events.
    # Coverage cannot be established automatically, so zero counts are marked unavailable.
    return dict(counts), "Local dated event CSV (30-day window); verify export completeness and event-type filtering"


for kind, assets in ASSETS.items():
    for asset in assets[:CONFIG["max_enrichment_assets"]] if CONFIG["max_enrichment_assets"] else assets:
        key = (kind, str(asset.get("id", "")))
        result = {"Connection Mode": NOT_AVAILABLE, "Connection Mode Source": NOT_AVAILABLE,
                  "Last Modified (UTC)": asset.get("updatedAt") or NOT_AVAILABLE,
                  "Last Extract Refresh (UTC)": NOT_AVAILABLE,
                  "Extract Refresh Source": NOT_AVAILABLE,
                  "Lifetime View Count": NOT_AVAILABLE,
                  "Lifetime Traffic Source": NOT_AVAILABLE,
                  "View Count — Last 30 Days": NOT_AVAILABLE,
                  "30-Day Traffic Source": NOT_AVAILABLE}
        detail = {}
        if CONFIG["enrich_connection_mode"] or CONFIG["enrich_last_modified"]:
            try:
                detail = rest_asset(kind, asset)
            except Exception as exc:
                enrich_error(kind, asset, "asset_detail", exc)
        if CONFIG["enrich_last_modified"]:
            result["Last Modified (UTC)"] = detail.get("updatedAt") or asset.get("updatedAt") or NOT_AVAILABLE
        if CONFIG["enrich_connection_mode"]:
            result["Connection Mode"], result["Connection Mode Source"] = connection_mode(kind, asset, detail)
        if CONFIG["enrich_extract_refresh"]:
            result["Last Extract Refresh (UTC)"], result["Extract Refresh Source"] = extract_timestamp(kind, asset)
        if kind == "workbooks" and CONFIG["enrich_lifetime_views"]:
            try:
                result["Lifetime View Count"], result["Lifetime Traffic Source"] = workbook_views(asset)
            except Exception as exc:
                enrich_error(kind, asset, "workbook_views", exc)
        ENRICHED[key] = result
        if CONFIG["enrichment_pause_seconds"]:
            time.sleep(float(CONFIG["enrichment_pause_seconds"]))

if CONFIG["enrich_30day_views"]:
    try:
        TRAFFIC_30DAY, traffic_note = load_30day_events()
    except Exception as exc:
        traffic_note = f"Not Available — CSV error: {exc}"
        ERRORS.append({"stage":"admin_insights_traffic_csv", "error":str(exc)})
    for (kind, asset_id), record in ENRICHED.items():
        if kind == "workbooks":
            if asset_id in TRAFFIC_30DAY:
                record["View Count — Last 30 Days"] = TRAFFIC_30DAY[asset_id]
                record["30-Day Traffic Source"] = traffic_note
            else:
                record["30-Day Traffic Source"] = traffic_note
print("Enrichment completed:", len(ENRICHED), "assets;", len(VIEW_TRAFFIC), "view rows;")

In [ ]:
def clean_text(value):
    if isinstance(value, dict):
        return value.get("name") or value.get("id") or json.dumps(value, ensure_ascii=False, default=str)
    if isinstance(value, list):
        return ", ".join(clean_text(v) for v in value)
    return "" if value is None else str(value)

def value_from(asset, *keys):
    for key in keys:
        if key in asset and asset[key] not in ("", None):
            return clean_text(asset[key])
    return ""

def asset_detail(kind, asset):
    if not CONFIG["fetch_asset_details"]:
        return {}
    singular = {"workbooks": "workbooks", "datasources": "datasources", "flows": "flows"}[kind]
    endpoint = f"/sites/{client.site_id}/{singular}/{asset['id']}"
    result = optional_get(client, endpoint)
    node = result.get(kind[:-1], {}) if isinstance(result, dict) else {}
    return node if isinstance(node, dict) else {}

def inventory_row(kind, asset, project_paths):
    detail = asset_detail(kind, asset)
    combined = {**asset, **detail}
    project_id = project_id_of(asset)
    owner = combined.get("owner") or {}
    if isinstance(owner, dict):
        owner_id = owner.get("id", "")
        owner_name = owner.get("name", "")
    else:
        owner_id, owner_name = str(owner), ""
    return {
        "Project Path": project_paths.get(project_id, "[Unresolved project]"),
        "Project Name": project_paths.get(project_id, "").split(" / ")[-1],
        "Asset Name": value_from(combined, "name"),
        "Asset Type": {"workbooks": "Workbook", "datasources": "Data Source", "flows": "Flow"}[kind],
        "Description": value_from(combined, "description"),
        "Project ID": project_id,
        "Asset ID": value_from(combined, "id", "luid"),
        "Owner Name": owner_name,
        "Owner ID": owner_id,
        "Created At": value_from(combined, "createdAt", "created_at"),
        "Updated At": value_from(combined, "updatedAt", "updated_at"),
        "Webpage URL": value_from(combined, "webpageUrl", "contentUrl"),
        "Content URL": value_from(combined, "contentUrl"),
        "Size / Type": value_from(combined, "size", "type", "contentType"),
        "Tags": clean_text(combined.get("tags", "")),
        **ENRICHED.get((kind, str(asset.get("id", ""))), {}),
        "Run ID": RUN_ID,
    }

def project_inventory_row(project):
    return {
        "Project Path": project.get("project_path", ""),
        "Project Name": project.get("name", ""),
        "Parent Project ID": clean_text(project.get("parentProjectId") or project.get("parent_project_id") or project.get("parentProject") or ""),
        "Project ID": project.get("id", ""),
        "Direct Name Match": "Yes" if project.get("direct_name_match") else "No — descendant",
        "Description": project.get("description", ""),
        "Owner ID": clean_text(project.get("owner", "")),
        "Run ID": RUN_ID,
    }

INVENTORY = {
    "Projects": sorted([project_inventory_row(p) for p in MATCHED_PROJECTS], key=lambda x:x["Project Path"].lower()),
}
for kind, title in (("workbooks","Workbooks"), ("datasources","Data Sources"), ("flows","Flows")):
    INVENTORY[title] = sorted(
        [inventory_row(kind, a, PROJECT_PATHS) for a in ASSETS[kind]],
        key=lambda x: (x["Project Path"].lower(), x["Asset Name"].lower())
    )
INVENTORY["All Assets"] = sorted(
    INVENTORY["Workbooks"] + INVENTORY["Data Sources"] + INVENTORY["Flows"],
    key=lambda x:(x["Project Path"].lower(), x["Asset Type"], x["Asset Name"].lower())
)
INVENTORY["View Traffic"] = VIEW_TRAFFIC
INVENTORY["Extraction Issues"] = list(ERRORS)

print("Inventory records:", {k:len(v) for k,v in INVENTORY.items()})

## Step 7A — Identify the preferred non-owner workbook viewers

**Optional input:** export Tableau Cloud Admin Insights **TS Events** (individual view-access events) and **TS Users** (site roles and identities) to CSV, then set `viewer_events_csv` and `viewer_users_csv` in Step 2. The notebook does not infer individual users from aggregate traffic.

For **each workbook**, identify **one most-frequent viewer** and **one most-recent viewer**. Exclude the workbook owner, then prioritize **Creator**, falling back to **Explorer** if no eligible Creator exists. Within that role tier, rank by view-access event count or latest event timestamp respectively. The two selected viewers may differ. When event/user identity or ownership cannot be verified, the cells explicitly show **Not Available**. These metrics reflect the time period covered by the exported events, not guaranteed lifetime history.

**Important:** Include `Item LUID`, `Item Type`, `Actor User ID`, `Event Type`, `Event Name`, `Event Date`, and `Event ID` in TS Events. Include `User ID`, `User LUID`, `User Site Role`, `User Friendly Name`, and `User Email` in TS Users. View events must be mappable to workbook LUIDs via the existing REST View Traffic inventory. Export view-access events only; do not include publishing/editing activity.

In [ ]:
# STEP 7A: Identify one preferred non-owner viewer per workbook for each criterion.
# Evidence: Admin Insights TS Events + TS Users. No invented viewer identities.
from collections import defaultdict
from datetime import datetime, timezone, timedelta
import pandas as pd

VIEWER_COLUMNS = [
    "Most Views — Viewer Name", "Most Views — Viewer Username", "Most Views — Viewer Site Role",
    "Most Views — Event Count", "Most Views — Last Viewed (UTC)",
    "Most Recent — Viewer Name", "Most Recent — Viewer Username", "Most Recent — Viewer Site Role",
    "Most Recent — Last Viewed (UTC)", "Most Recent — Event Count",
    "Viewer Selection Status", "Viewer Evidence Source", "Viewer Event Coverage",
]


def _norm_id(value):
    if value is None or pd.isna(value):
        return ""
    s = str(value).strip()
    return s[:-2] if s.endswith(".0") and s[:-2].isdigit() else s


def _column(frame, *names):
    lookup = {str(col).strip().casefold(): col for col in frame.columns}
    for name in names:
        key = str(name).strip().casefold()
        if key in lookup:
            return lookup[key]
    return None


def _safe_str(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


def _role_level(role):
    r = str(role).casefold().replace(" ", "")
    if "creator" in r:
        return 0
    if "explorer" in r:
        return 1
    return 2


def _load_users(path):
    if not path:
        return {}, {}, "No TS Users CSV configured"
    frame = pd.read_csv(Path(path).expanduser(), dtype=str, keep_default_na=False)
    id_col = _column(frame, "User ID", "User Id")
    luid_col = _column(frame, "User LUID", "User Luid")
    role_col = _column(frame, "User Site Role", "Site Role", "siteRole")
    name_col = _column(frame, "User Friendly Name", "User Name", "Name")
    email_col = _column(frame, "User Email", "Email", "User Name")
    if not role_col or not (id_col or luid_col):
        raise ValueError("TS Users CSV needs 'User Site Role' and 'User ID' or 'User LUID'.")
    by_id, by_email = {}, {}
    for _, row in frame.iterrows():
        info = {
            "id": _norm_id(row[id_col]) if id_col else "",
            "luid": _norm_id(row[luid_col]) if luid_col else "",
            "name": _safe_str(row[name_col]) if name_col else "",
            "email": _safe_str(row[email_col]).casefold() if email_col else "",
            "role": _safe_str(row[role_col]),
        }
        for uid in (info["id"], info["luid"]):
            if uid: by_id[uid] = info
        if info["email"]: by_email[info["email"]] = info
    return by_id, by_email, f"TS Users CSV ({len(frame)} records)"


def _load_viewer_events(path, users_by_id, users_by_email, workbooks):
    if not path:
        return [], "No TS Events CSV configured"
    frame = pd.read_csv(Path(path).expanduser(), dtype=str, keep_default_na=False, low_memory=False)
    date_col = _column(frame, "Event Date")
    actor_id_col = _column(frame, "Actor User ID", "Actor User Id")
    actor_name_col = _column(frame, "Actor User Name")
    item_type_col = _column(frame, "Item Type")
    item_luid_col = _column(frame, "Item LUID")
    event_type_col = _column(frame, "Event Type")
    event_name_col = _column(frame, "Event Name")
    workbook_luid_col = _column(frame, "Workbook LUID", "Workbook ID")
    view_luid_col = _column(frame, "View LUID", "View ID")
    if not date_col or not (actor_id_col or actor_name_col) or not item_type_col:
        raise ValueError("TS Events CSV needs Event Date, Item Type, and Actor User ID or Actor User Name.")
    if not (item_luid_col or workbook_luid_col or view_luid_col):
        raise ValueError("TS Events CSV needs Item LUID or a verified Workbook/View LUID.")
    workbook_ids = {_norm_id(r.get("Asset ID")) for r in workbooks}
    view_map = {_norm_id(v.get("View ID")): _norm_id(v.get("Workbook ID"))
                for v in globals().get("VIEW_TRAFFIC", []) if v.get("View ID") and v.get("Workbook ID")}
    # The workbook view IDs come from REST, not workbook-name guesses.
    events = []
    dropped_unmatched = dropped_unresolved = 0
    cutoff = None
    if CONFIG.get("viewer_lookback_days") is not None:
        cutoff = datetime.now(timezone.utc) - timedelta(days=int(CONFIG["viewer_lookback_days"]))
    for _, row in frame.iterrows():
        kind = _safe_str(row[item_type_col]).casefold()
        if kind not in ("view", "workbook"): continue
        etype = _safe_str(row[event_type_col]).casefold() if event_type_col else ""
        ename = _safe_str(row[event_name_col]).casefold() if event_name_col else ""
        allowed_types = {str(v).casefold() for v in CONFIG["viewer_event_types"]}
        keywords = tuple(str(v).casefold() for v in CONFIG["viewer_event_name_keywords"])
        # Strict access filter: exclude editing, publishing, downloads, logins, etc.
        is_access = etype in allowed_types
        named_access = any(k in ename for k in keywords)
        if not (is_access and (named_access or (not ename and CONFIG["viewer_allow_access_type_without_event_name"]))):
            continue
        dt = pd.to_datetime(row[date_col], utc=True, errors="coerce")
        if pd.isna(dt): continue
        stamp = dt.to_pydatetime()
        if cutoff and stamp < cutoff: continue
        workbook_id = _norm_id(row[workbook_luid_col]) if workbook_luid_col else ""
        item_id = _norm_id(row[item_luid_col]) if item_luid_col else ""
        view_id = _norm_id(row[view_luid_col]) if view_luid_col else ""
        if not workbook_id:
            if kind == "workbook": workbook_id = item_id
            else: workbook_id = view_map.get(view_id or item_id, "")
        if not workbook_id or workbook_id not in workbook_ids:
            dropped_unmatched += 1
            continue
        actor_id = _norm_id(row[actor_id_col]) if actor_id_col else ""
        actor_name = _safe_str(row[actor_name_col]).casefold() if actor_name_col else ""
        user = users_by_id.get(actor_id) or users_by_email.get(actor_name)
        if not user or not user.get("role"):
            dropped_unresolved += 1
            continue
        events.append({"workbook":workbook_id,"user":user,"date":stamp,
                       "event_id":_safe_str(row[_column(frame,"Event ID")]) if _column(frame,"Event ID") else ""})
    # Deduplicate by actual Event ID, if available (not by same timestamp).
    seen, unique = set(), []
    for e in events:
        k = e["event_id"]
        if k and k in seen: continue
        if k: seen.add(k)
        unique.append(e)
    return unique, (f"TS Events CSV: {len(unique)} matched access events; "
                    f"{dropped_unmatched} not mapped to inventory workbooks; "
                    f"{dropped_unresolved} actors not resolved")


def _owner_identifiers(asset_id):
    asset = next((a for a in ASSETS.get("workbooks", []) if _norm_id(a.get("id")) == asset_id), {})
    owner = asset.get("owner") or {}
    if not isinstance(owner, dict): owner = {"id": owner}
    return {_norm_id(owner.get("id")), _norm_id(owner.get("name")).casefold(),
            _norm_id(owner.get("email")).casefold() } - {""}


def _select_viewers(workbook_id, events):
    # One individual for highest frequency, and one for most recent activity.
    # Creator > Explorer preference is applied BEFORE usage/recency ranking.
    owners = _owner_identifiers(workbook_id)
    candidates = {}
    for e in events:
        u = e["user"]
        identities = {u.get("id", ""), u.get("luid", ""), u.get("email", ""),
                      u.get("name", "").casefold()} - {""}
        if owners and owners.intersection(identities): continue
        # Fail closed when owner identity is unavailable: cannot verify non-owner.
        if not owners: continue
        level = _role_level(u["role"])
        if level == 2 and not CONFIG.get("viewer_allow_other_roles", False): continue
        key = u.get("luid") or u.get("id") or u.get("email")
        if not key: continue
        entry = candidates.setdefault(key, {"user":u, "count":0, "last":e["date"], "level":level})
        entry["count"] += 1
        entry["last"] = max(entry["last"],e["date"])
    if not candidates:
        return None,None,"No eligible verified non-owner Creator/Explorer viewer"
    eligible = [v for v in candidates.values() if v["level"] == min(v["level"] for v in candidates.values())]
    most = sorted(eligible, key=lambda v:(-v["count"], -v["last"].timestamp(),v["user"].get("email", "")))[0]
    recent = sorted(eligible, key=lambda v:(-v["last"].timestamp(), -v["count"],v["user"].get("email", "")))[0]
    return most,recent,"Selected; role preference Creator > Explorer; owner excluded"


# Workbook-level non-owner consumption evidence, independent of Creator/Explorer
# preference used to select the named "Most Views" / "Most Recent" individuals.
NON_OWNER_COLUMNS = [
    "Non-Owner Views — Last 30 Days", "Unique Non-Owner Viewers — Last 30 Days",
    "Last Non-Owner View (UTC)", "Non-Owner Views — Evidence Status",
    "Non-Owner Views — Evidence Source",
]

def _non_owner_aggregates(workbook_id, events, export_coverage):
    owners = _owner_identifiers(workbook_id)
    if not owners:
        return None, None, None, "Unknown — workbook owner cannot be verified"
    if not export_coverage:
        return None, None, None, "Unknown — TS Events coverage window unverified"
    now = datetime.now(timezone.utc)
    days = max(1, int(CONFIG.get("non_owner_view_lookback_days", 30)))
    cutoff = now - timedelta(days=days)
    verified = []
    distinct = set()
    for e in events:
        u = e["user"]
        identities = {str(u.get(k) or "").strip().casefold() for k in ("id", "luid", "email", "name")} - {""}
        if identities & {str(v).casefold() for v in owners}:
            continue
        if not cutoff <= e["date"] <= now:
            continue
        key = u.get("luid") or u.get("id") or u.get("email")
        if not key:
            continue
        verified.append(e)
        distinct.add(key)
    last = max((e["date"] for e in verified), default=None)
    return len(verified), len(distinct), last.isoformat() if last else "Not Available", "Verified — complete export window"


def _export_coverage_complete(path):
    # A CSV alone cannot prove that all Tableau access events were exported.
    # Explicitly require an operator-provided coverage confirmation flag.
    return bool(CONFIG.get("non_owner_events_coverage_confirmed", False))

def enrich_inventory_viewers():
    records = INVENTORY.get("Workbooks", [])
    users_path = str(CONFIG.get("viewer_users_csv") or "").strip()
    events_path = str(CONFIG.get("viewer_events_csv") or "").strip()
    status = "Not Available — provide both TS Events and TS Users CSV exports"
    selected = {}
    aggregates = {}
    if users_path and events_path:
        try:
            users_by_id,users_by_email,user_note = _load_users(users_path)
            events,event_note = _load_viewer_events(events_path,users_by_id,users_by_email,records)
            grouped = defaultdict(list)
            for e in events: grouped[e["workbook"]].append(e)
            complete = _export_coverage_complete(events_path)
            for record in records:
                aid = _norm_id(record.get("Asset ID"))
                selected[aid] = _select_viewers(aid,grouped.get(aid,[]))
                aggregates[aid] = _non_owner_aggregates(aid,grouped.get(aid,[]),complete)
            status = user_note + "; " + event_note
        except Exception as exc:
            status = f"Not Available — viewer enrichment error: {exc}"
            ERRORS.append({"stage":"viewer_identification","error":str(exc)})
    for sheet in ("Workbooks","All Assets"):
        for record in INVENTORY.get(sheet,[]):
            if record.get("Asset Type") != "Workbook": continue
            aid = _norm_id(record.get("Asset ID"))
            most,recent,selection = selected.get(aid,(None,None,"Not Available — missing usable user-level access evidence"))
            record.update({col:"Not Available" for col in VIEWER_COLUMNS})
            record.update({col:"Not Available" for col in NON_OWNER_COLUMNS})
            count, unique_count, last_non_owner, evidence_status = aggregates.get(
                aid, (None, None, None, "Unknown — missing or incomplete TS Events/TS Users evidence"))
            record["Non-Owner Views — Evidence Status"] = evidence_status
            record["Non-Owner Views — Evidence Source"] = status
            if count is not None:
                record["Non-Owner Views — Last 30 Days"] = count
                record["Unique Non-Owner Viewers — Last 30 Days"] = unique_count
                record["Last Non-Owner View (UTC)"] = last_non_owner
            for prefix,entry in (("Most Views",most),("Most Recent",recent)):
                if not entry: continue
                u = entry["user"]
                record[f"{prefix} — Viewer Name"] = u.get("name") or u.get("email") or "Not Available"
                record[f"{prefix} — Viewer Username"] = u.get("email") or "Not Available"
                record[f"{prefix} — Viewer Site Role"] = u.get("role") or "Not Available"
                if prefix == "Most Views":
                    record["Most Views — Event Count"] = entry["count"]
                    record["Most Views — Last Viewed (UTC)"] = entry["last"].isoformat()
                else:
                    record["Most Recent — Event Count"] = entry["count"]
                    record["Most Recent — Last Viewed (UTC)"] = entry["last"].isoformat()
            record["Viewer Selection Status"] = selection if selected else status
            record["Viewer Evidence Source"] = status
            record["Viewer Event Coverage"] = "Export coverage only; not guaranteed lifetime"
    print("Viewer enrichment:",status)
    print("Workbooks with identified top viewer:",sum(
        r.get("Most Views — Viewer Name") != "Not Available" for r in records))

enrich_inventory_viewers()


## Step 8 — Sign out safely
 Close the Tableau session after inventory collection. The following export step is entirely local and does not require a live Tableau connection.

In [ ]:
try:
    client.signout()
except Exception as exc:
    log.warning("Signout failed: %s", exc)
finally:
    client.session.close()
print("Tableau session closed.")

## Step 9 — Export styled Inventory Excel
 Sheets: **Overview, Projects, Workbooks, Data Sources, Flows, All Assets, View Traffic, Extraction Issues**. Features: frozen headers, column sizing, dark-blue headings, color-coded asset types, conditional formatting, and Excel row/column outline groups. Row groups are by contiguous project path; technical columns can be expanded/collapsed using Excel **Data → Group/Ungroup**. Also saves a JSON run manifest.

**Hyperlink navigation:** Overview quick links; Projects → All Assets; All Assets ↔ type-specific sheets; each asset → parent project; View Traffic → Workbooks; and Tableau Cloud links when a valid URL is available. Asset relationships are matched by LUID, not by similar names.

**Activity triage:** `Activity Priorities` sorts assets by P1–P5 and highlights usage evidence. Recent modification and extract refresh are separate from actual views. Missing traffic is never treated as zero.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.formatting.rule import FormulaRule
from openpyxl.utils import get_column_letter

MAX_EXCEL_ROWS = 1048576

def as_cell(value):
    if value is None:
        return ""
    if isinstance(value, (dict, list)):
        value = json.dumps(value, ensure_ascii=False, default=str)
    if isinstance(value, (bool, int, float)):
        return value
    text = str(value).replace("\x00", "")[:32000]
    # Avoid Excel interpreting untrusted Tableau names as formulas.
    if text.startswith(("=", "+", "-", "@")):
        text = "'" + text
    return text

def contiguous_project_groups(rows):
    groups = []
    if not rows:
        return groups
    current = rows[0].get("Project Path", "")
    first = 2
    for idx, item in enumerate(rows[1:], start=3):
        new = item.get("Project Path", "")
        if new != current:
            if idx - 1 > first:
                groups.append((first, idx - 1))
            first, current = idx, new
    if len(rows) + 1 > first:
        groups.append((first, len(rows) + 1))
    return groups

def add_sheet(wb, name, rows):
    sheet = wb.create_sheet(title=name)
    columns = list(dict.fromkeys(k for row in rows for k in row.keys())) if rows else ["Information"]
    if not columns:
        columns = ["Information"]
    sheet.append(columns)
    for row in rows[:MAX_EXCEL_ROWS - 1]:
        sheet.append([as_cell(row.get(col, "")) for col in columns])
    sheet.freeze_panes = "A2"
    sheet.sheet_view.showGridLines = False
    sheet.sheet_properties.pageSetUpPr.fitToPage = True
    sheet.print_options.horizontalCentered = True
    sheet.print_title_rows = "1:1"
    sheet.row_dimensions[1].height = 30
    for c in sheet[1]:
        c.fill = PatternFill("solid", fgColor="17365D")
        c.font = Font(name="Aptos", size=10, bold=True, color="FFFFFF")
        c.alignment = Alignment(vertical="center", wrap_text=True)
    for j, col in enumerate(columns, 1):
        width = 54 if col in ("Project Path", "Description") else 37 if col in ("Asset Name", "Webpage URL") else 23
        sheet.column_dimensions[get_column_letter(j)].width = width
    for row in sheet.iter_rows(min_row=2):
        for c in row:
            c.font = Font(name="Aptos", size=10, color="273449")
            c.alignment = Alignment(vertical="center", wrap_text=False)
    if sheet.max_row > 1:
        sheet.auto_filter.ref = f"A1:{get_column_letter(len(columns))}{sheet.max_row}"
        if name == "Projects" and "Direct Name Match" in columns:
            col = get_column_letter(columns.index("Direct Name Match") + 1)
            sheet.conditional_formatting.add(
                f"{col}2:{col}{sheet.max_row}",
                FormulaRule(formula=[f'{col}2="Yes"'],
                            fill=PatternFill("solid", fgColor="DCFCE7"),
                            font=Font(color="166534", bold=True))
            )
        if name == "Extraction Issues":
            for row in sheet.iter_rows(min_row=2):
                for c in row:
                    c.fill = PatternFill("solid", fgColor="FFF1F2")
        if name in ("Workbooks", "Data Sources", "Flows", "All Assets", "Activity Priorities") and "Asset Type" in columns:
            col = get_column_letter(columns.index("Asset Type") + 1)
            for typ, fill, font in (
                ("Workbook", "DBEAFE", "1D4ED8"),
                ("Data Source", "D1FAE5", "065F46"),
                ("Flow", "FEF3C7", "92400E"),
            ):
                sheet.conditional_formatting.add(
                    f"{col}2:{col}{sheet.max_row}",
                    FormulaRule(formula=[f'{col}2="{typ}"'],
                                fill=PatternFill("solid", fgColor=fill),
                                font=Font(color=font, bold=True))
                )
    if name in ("Workbooks", "Data Sources", "Flows", "All Assets", "Activity Priorities") and rows:
        for _field, _mapping in {
            "Activity Priority": [
                ("P1", "FECACA", "991B1B"), ("P2", "FED7AA", "9A3412"),
                ("P3", "FEF08A", "854D0E"), ("P4", "DBEAFE", "1E40AF"),
                ("P5", "E5E7EB", "4B5563"),
            ],
            "Activity Status": [
                ("Recently Viewed", "DCFCE7", "166534"),
                ("Recently Maintained", "FEF9C3", "854D0E"),
                ("Activity Unknown", "E5E7EB", "4B5563"),
            ],
        }.items():
            if _field in columns:
                _letter = get_column_letter(columns.index(_field) + 1)
                for _prefix, _fill, _font in _mapping:
                    sheet.conditional_formatting.add(
                        f"{_letter}2:{_letter}{sheet.max_row}",
                        FormulaRule(formula=[f'LEFT(${_letter}2,{len(_prefix)})="{_prefix}"'],
                                    fill=PatternFill("solid", fgColor=_fill),
                                    font=Font(color=_font, bold=True)))
    if name in ("Workbooks", "Data Sources", "Flows", "All Assets", "Activity Priorities", "Projects"):
        essential_columns = 5 if name == "Projects" else 7
        if CONFIG["group_technical_columns"] and len(columns) > essential_columns:
            sheet.column_dimensions.group(get_column_letter(essential_columns + 1),
                                          get_column_letter(len(columns)),
                                          outline_level=1, hidden=False)
        if CONFIG["group_rows_by_project"]:
            for start, end in contiguous_project_groups(rows):
                sheet.row_dimensions.group(start, end, outline_level=1, hidden=False)
        sheet.sheet_properties.outlinePr.summaryBelow = True
        sheet.sheet_properties.outlinePr.summaryRight = True
    return sheet

SUMMARY_ROWS = [
    {"Metric":"Extraction run ID","Value":RUN_ID},
    {"Metric":"Tableau site","Value":CONFIG["tableau_server"]},
    {"Metric":"Site content URL","Value":CONFIG["site_content_url"]},
    {"Metric":"Project name contains (case-insensitive)","Value":CONFIG["project_search"]},
    {"Metric":"Include descendant projects","Value":str(CONFIG["include_descendant_projects"])},
    {"Metric":"Visible projects","Value":len(ALL_PROJECTS)},
    {"Metric":"Direct matching projects","Value":len(DIRECT_MATCH_IDS)},
    {"Metric":"Included projects","Value":len(PROJECT_IDS)},
    {"Metric":"Workbooks","Value":len(INVENTORY["Workbooks"])},
    {"Metric":"Published data sources","Value":len(INVENTORY["Data Sources"])},
    {"Metric":"Flows","Value":len(INVENTORY["Flows"])},
    {"Metric":"Extraction warnings","Value":len(ERRORS)},
    {"Metric":"Metadata depth","Value":"Project + asset inventory (no KPI/field/formula extraction)"},
    {"Metric":"Coverage note","Value":"Only assets visible to the signed-in PAT account"},
]
# ---------------------------------------------------------------------------
# Evidence-based activity classification and triage. No inferred zero traffic.
# ---------------------------------------------------------------------------
from datetime import datetime as _activity_datetime, timezone as _activity_timezone, timedelta as _activity_timedelta

def _activity_number(value):
    if value is None or isinstance(value, bool): return None
    try:
        if str(value).strip().lower() in ("", "not available", "none", "nan", "unknown"):
            return None
        result = float(str(value).replace(",", ""))
        return result if result >= 0 and result != float("inf") else None
    except (ValueError, TypeError):
        return None

def _activity_date(value):
    if not value or str(value).strip().lower() in ("not available", "unknown", "none"):
        return None
    try:
        if isinstance(value, _activity_datetime):
            parsed = value
        else:
            parsed = _activity_datetime.fromisoformat(str(value).strip().replace("Z", "+00:00"))
        if parsed.tzinfo is None: parsed = parsed.replace(tzinfo=_activity_timezone.utc)
        return parsed.astimezone(_activity_timezone.utc)
    except (ValueError, TypeError):
        return None

def classify_activity(record, now=None):
    """Evidence-first ranking. Verified non-owner use outranks maintenance.

    Non-owner counts are only considered verified if the export's 30-day
    completeness has been confirmed by the operator. Otherwise the count is
    unknown, not zero; overall traffic remains visible as context only.
    """
    now = now or _activity_datetime.now(_activity_timezone.utc)
    days = max(1, int(CONFIG["recent_activity_days"]))
    mod_days = max(1, int(CONFIG["recent_modification_days"]))
    overall30 = _activity_number(record.get("View Count — Last 30 Days"))
    non_owner30 = _activity_number(record.get("Non-Owner Views — Last 30 Days"))
    unique_non_owner = _activity_number(record.get("Unique Non-Owner Viewers — Last 30 Days"))
    last_non_owner = _activity_date(record.get("Last Non-Owner View (UTC)"))
    status_evidence = str(record.get("Non-Owner Views — Evidence Status", ""))
    verified = status_evidence.startswith("Verified —") and non_owner30 is not None
    lifetime = _activity_number(record.get("Lifetime View Count"))
    modified = _activity_date(record.get("Last Modified (UTC)")) or _activity_date(record.get("Updated At"))
    refreshed = _activity_date(record.get("Last Extract Refresh (UTC)"))
    modified_recent = bool(modified and now - _activity_timedelta(days=mod_days) <= modified <= now)
    refresh_recent = bool(refreshed and now - _activity_timedelta(days=days) <= refreshed <= now)
    evidence = []
    if verified:
        evidence.append(f"Verified non-owner views in last {days} days: {non_owner30:g}; unique viewers: {unique_non_owner:g}")
    else:
        evidence.append("Non-owner view count unknown (unverified event coverage / ownership)")
    if overall30 is not None: evidence.append(f"Overall 30-day views (context only): {overall30:g}")
    if lifetime is not None: evidence.append(f"Lifetime views (undated): {lifetime:g}")
    if modified_recent: evidence.append(f"Modified within {mod_days} days")
    if refresh_recent: evidence.append(f"Extract refreshed within {days} days")
    if CONFIG.get("prioritize_verified_non_owner_views", True):
        if verified and non_owner30 >= int(CONFIG["non_owner_high_traffic_threshold"]):
            status, priority, rank = "Verified Non-Owner Usage — High", "P1 — High Non-Owner Usage", 1
        elif verified and non_owner30 > 0:
            status, priority, rank = "Verified Non-Owner Usage — Recent", "P2 — Non-Owner Usage", 2
        elif modified_recent or refresh_recent:
            status, priority, rank = "Recently Maintained / Refreshed", "P3 — Recent Change", 3
        elif verified and non_owner30 == 0:
            status, priority, rank = "Verified Zero Non-Owner Views", "P4 — Review", 4
        else:
            status, priority, rank = "Non-Owner Usage Unknown", "P5 — Insufficient Evidence", 5
    else:
        if overall30 is not None and overall30 >= int(CONFIG["high_traffic_30day_threshold"]):
            status, priority, rank = "Recently Viewed", "P1 — High Usage", 1
        elif overall30 is not None and overall30 > 0:
            status, priority, rank = "Recently Viewed", "P2 — Used Recently", 2
        elif modified_recent or refresh_recent:
            status, priority, rank = "Recently Maintained / Refreshed", "P3 — Recent Change", 3
        elif overall30 == 0:
            status, priority, rank = "No Recorded Views in 30 Days", "P4 — Review", 4
        else:
            status, priority, rank = "Activity Unknown", "P5 — Insufficient Evidence", 5
    observed = max([d for d in (last_non_owner, modified, refreshed) if d], default=None)
    return {
        "Activity Status": status,
        "Activity Priority": priority,
        "Activity Rank": rank,
        "Activity Evidence": "; ".join(evidence),
        "Last Observed Activity (UTC)": observed.isoformat() if observed else "Not Available",
        "Non-Owner Consumption Verified": "Yes" if verified else "No",
    }

if CONFIG.get("enable_activity_prioritization", True):
    # Classify once per asset ID; same classification on All Assets and individual tabs.
    for _sheet_name in ("Workbooks", "Data Sources", "Flows", "All Assets"):
        for _record in INVENTORY.get(_sheet_name, []):
            _record.update(classify_activity(_record))
    _priority_rows = [dict(r) for r in INVENTORY.get("All Assets", [])]
    _priority_rows.sort(key=lambda r: (r.get("Activity Rank", 5),
                                      -(_activity_number(r.get("Non-Owner Views — Last 30 Days")) or 0),
                                      -(_activity_number(r.get("Unique Non-Owner Viewers — Last 30 Days")) or 0),
                                      str(r.get("Project Path", "")).casefold(),
                                      str(r.get("Asset Name", "")).casefold()))
    INVENTORY[CONFIG["priority_sheet_name"]] = _priority_rows
    SUMMARY_ROWS.extend([
        {"Metric":"Recently viewed assets (dated 30-day evidence)","Value":sum(r["Activity Rank"] in (1, 2) for r in _priority_rows)},
        {"Metric":"Recently maintained / refreshed assets","Value":sum(r["Activity Status"] == "Recently Maintained / Refreshed" for r in _priority_rows)},
        {"Metric":"Activity unknown assets","Value":sum(r["Activity Rank"] == 5 for r in _priority_rows)},
        {"Metric":"Workbooks with verified non-owner usage", "Value":sum(r.get("Non-Owner Consumption Verified") == "Yes" for r in _priority_rows)},
        {"Metric":"Activity classification caveat","Value":"Modification / refresh does not prove user views; missing 30-day traffic is not zero."},
    ])

INVENTORY = {"Overview":SUMMARY_ROWS, **INVENTORY}
wb = Workbook()
wb.remove(wb.active)
for name, rows in INVENTORY.items():
    add_sheet(wb, name, rows)

# ---------------------------------------------------------------------------
# Cross-sheet navigation and verified Tableau Cloud asset links
# ---------------------------------------------------------------------------
from urllib.parse import urlparse, quote
from openpyxl.worksheet.hyperlink import Hyperlink

SHEET_FOR_TYPE = {"Workbook": "Workbooks", "Data Source": "Data Sources", "Flow": "Flows"}


def safe_cloud_url(raw):
    """Only link to explicit absolute Tableau URLs; do not guess asset routes."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    value = raw.strip()
    parsed = urlparse(value)
    server = urlparse(str(CONFIG["tableau_server"]).rstrip("/"))
    if parsed.scheme != "https" or parsed.hostname != server.hostname:
        return ""
    return value


def set_internal_link(cell, destination_sheet, destination_row, label=None):
    """Use native OOXML internal links (not external URLs with a # fragment)."""
    if label is not None:
        cell.value = label
    destination = "'" + destination_sheet.replace("'", "''") + "'!A" + str(destination_row)
    cell.hyperlink = Hyperlink(ref=cell.coordinate, location=destination,
                               display=str(cell.value), tooltip=f"Go to {destination_sheet} row {destination_row}")
    cell.font = Font(name="Aptos", size=10, color="0563C1", underline="single")


def set_external_link(cell, url, label="Open Tableau"):
    verified = safe_cloud_url(url)
    if verified:
        cell.value = label
        cell.hyperlink = verified
        cell.font = Font(name="Aptos", size=10, color="0563C1", underline="single")
    else:
        cell.value = "Not Available"


def add_inventory_navigation(workbook, inventory):
    """Link records by exact LUID, never by potentially duplicated asset names."""
    if not CONFIG.get("enable_excel_hyperlinks", True):
        return
    asset_locations = {}
    project_locations = {}
    for sheet_name in ("Projects", "Workbooks", "Data Sources", "Flows", "All Assets", "Activity Priorities"):
        ws = workbook[sheet_name]
        rows = inventory.get(sheet_name, [])
        for excel_row, record in enumerate(rows, 2):
            if sheet_name == "Projects":
                pid = str(record.get("Project ID") or "")
                if pid:
                    project_locations[pid] = excel_row
            else:
                asset_id = str(record.get("Asset ID") or "")
                typ = str(record.get("Asset Type") or "")
                if asset_id and typ:
                    asset_locations[(sheet_name, typ, asset_id)] = excel_row

    # Add explicit navigation columns without disturbing existing column positions.
    for sheet_name in ("Projects", "Workbooks", "Data Sources", "Flows", "All Assets", "Activity Priorities", "View Traffic"):
        ws = workbook[sheet_name]
        records = inventory.get(sheet_name, [])
        if not records:
            continue
        labels = (["Project Assets", "Open in Tableau Cloud"] if sheet_name == "Projects"
                  else ["Related Excel Record", "Project Folder", "Open in Tableau Cloud"]
                  if sheet_name in ("Workbooks", "Data Sources", "Flows", "All Assets", "Activity Priorities")
                  else ["Workbook Record"])
        start_col = ws.max_column + 1
        for offset, title in enumerate(labels):
            cell = ws.cell(1, start_col + offset, title)
            cell.fill = PatternFill("solid", fgColor="17365D")
            cell.font = Font(name="Aptos", size=10, bold=True, color="FFFFFF")
            cell.alignment = Alignment(vertical="center", wrap_text=True)
            ws.column_dimensions[get_column_letter(start_col + offset)].width = 26
        for excel_row, record in enumerate(records, 2):
            cells = [ws.cell(excel_row, start_col + i) for i in range(len(labels))]
            if sheet_name == "Projects":
                pid = str(record.get("Project ID") or "")
                targets = [(i,r) for i,r in enumerate(inventory.get("All Assets", []),2)
                           if str(r.get("Project ID") or "") == pid]
                if targets:
                    set_internal_link(cells[0], "All Assets", targets[0][0], f"Assets ({len(targets)})")
                else:
                    cells[0].value = "No direct assets"
                # A project URL is deterministic from the verified Tableau site and its actual project LUID.
                if CONFIG.get("enable_tableau_cloud_links", True) and pid:
                    url = (str(CONFIG["tableau_server"]).rstrip("/") + "/#/site/" +
                           quote(str(CONFIG["site_content_url"]).strip("/"), safe="") +
                           "/projects/" + quote(pid, safe=""))
                    set_external_link(cells[1], url, "Open Project")
                continue
            if sheet_name == "View Traffic":
                aid = str(record.get("Workbook ID") or "")
                row = asset_locations.get(("Workbooks", "Workbook", aid))
                if row:
                    set_internal_link(cells[0], "Workbooks", row, "Go to Workbook")
                else:
                    cells[0].value = "Not in inventory"
                continue
            typ = str(record.get("Asset Type") or "")
            aid = str(record.get("Asset ID") or "")
            target_sheet = (SHEET_FOR_TYPE.get(typ) if sheet_name == "All Assets" else "All Assets")
            target_row = asset_locations.get((target_sheet, typ, aid)) if target_sheet else None
            if target_row:
                set_internal_link(cells[0], target_sheet, target_row, f"Go to {target_sheet}")
            else:
                cells[0].value = "Not Available"
            pid = str(record.get("Project ID") or "")
            prow = project_locations.get(pid)
            if prow:
                set_internal_link(cells[1], "Projects", prow, "Go to Project")
            else:
                cells[1].value = "Not Available"
            if CONFIG.get("enable_tableau_cloud_links", True):
                # 'Content URL' is often relative; only the actual full webpage URL is safe.
                set_external_link(cells[2], record.get("Webpage URL", ""))
            else:
                cells[2].value = "Disabled"
        ws.auto_filter.ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"
    # Overview navigation for instant entry into every inventory tab.
    overview = workbook["Overview"]
    nav_col = overview.max_column + 2
    heading = overview.cell(1, nav_col, "Quick Navigation")
    heading.fill = PatternFill("solid", fgColor="17365D")
    heading.font = Font(name="Aptos", bold=True, color="FFFFFF")
    overview.column_dimensions[get_column_letter(nav_col)].width = 29
    for i, name in enumerate(("Projects", "Workbooks", "Data Sources", "Flows", "All Assets", "Activity Priorities", "View Traffic", "Extraction Issues"), 2):
        set_internal_link(overview.cell(i, nav_col), name, 1, f"Open {name}")
    return {"linked_assets": len(asset_locations), "linked_projects": len(project_locations)}

LINK_SUMMARY = add_inventory_navigation(wb, INVENTORY)

output_path = ROOT / CONFIG["inventory_filename"]
wb.save(output_path)
manifest = {
    "run_id":RUN_ID, "generated_utc":datetime.now(timezone.utc).isoformat(),
    "search_phrase":CONFIG["project_search"],
    "included_descendants":CONFIG["include_descendant_projects"],
    "included_project_ids":sorted(PROJECT_IDS),
    "inventory_counts":{name:len(rows) for name,rows in INVENTORY.items()},
    "warnings":ERRORS,
    "hyperlink_summary":LINK_SUMMARY,
    "activity_policy": {k:CONFIG[k] for k in ("recent_activity_days", "recent_modification_days", "high_traffic_30day_threshold", "medium_traffic_30day_threshold")},
    "workbook_path":str(output_path),
    "limitations":[
        "Inventory reflects only projects and assets visible to the PAT.",
        "No workbook XML, Tableau Prep node graph, KPI, field, formula or metric extraction.",
        "Flow endpoint may be unavailable on some sites or permissions.",
        "Project/asset metadata fields vary by Tableau Cloud API and asset type.",
        "Rows are grouped by contiguous project path, and extra metadata columns are expandable.",
    ],
}
(ROOT/"run_manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
print("Inventory Excel:",output_path)
print("Manifest:",ROOT/"run_manifest.json")
print("Counts:",manifest["inventory_counts"])

## Non-owner usage prioritization — configuration and evidence\n\nPriority uses verified non-owner workbook view events from Tableau Admin Insights TS Events and TS Users. Configure both CSV paths and set `non_owner_events_coverage_confirmed=True` **only after verifying that the exports cover the full last 30 days**. Without that confirmation, counts remain unknown and cannot be treated as zero. Workbook ownership must be resolvable.\n\nP1: >=100 verified non-owner views; P2: 1–99; P3: recently modified/refreshed when no positive verified consumption; P4: verified zero views without recent maintenance; P5: insufficient evidence. Overall views remain context only. Non-workbook assets are prioritized on maintenance/refresh evidence and are not assigned invented non-owner counts.\n

## Step 10 — Explore the inventory
 Open `Commercial_Finance_Project_Inventory.xlsx` from the configured output folder. Filter and sort by project path, asset type and asset name. Expand the technical columns when needed; use **Data → Ungroup** to remove grouping. This intentionally excludes granular field/calculation/KPI documentation to keep discovery quick.

 **Notes:** Inventory coverage depends on your PAT's access. Some REST fields may be absent or unsupported. Review the `Extraction Issues` sheet and `run_manifest.json` for limitations.

**Hyperlinks:** Blue underlined links navigate within Excel. Tableau Cloud links require sign-in and are only created for supported URLs. This inventory does not infer unverified workbook–data-source dependencies.

Start with **Activity Priorities**. P1/P2 means confirmed recent view events; P3 means recently modified/refreshed but not necessarily viewed; P4 means historical or confirmed zero activity; P5 means insufficient evidence. The classification is intentionally conservative.

**Viewer columns:** Workbooks, All Assets, and Activity Priorities include most-frequent and most-recent eligible non-owner viewers, their roles and evidence status. Missing Admin Insights exports produce Not Available values.